# S1 Staging ? FX Rates

Parse source dates and rates, validate date-currency keys, and preserve the original rate series. No dates are filled and no currency conversion is performed.


# 1. Imports & Configuration


In [ ]:
import polars as pl
from pathlib import Path

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "raw_data").is_dir() and (path / "s1_staging").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the FABINI project root from the current directory.")

RAW_DIR = PROJECT_ROOT / "raw_data"
OUTPUT_DIR = PROJECT_ROOT / "s1_staging" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_FILE = "raw_fx_rates.csv"
OUTPUT_PATH = OUTPUT_DIR / "fx_rates.parquet"
# Expected counts for the supplied case-study export, not general source constraints.
CASE_STUDY_EXPECTED_ROWS = 382
CASE_STUDY_EXPECTED_ROWS_PER_CURRENCY = 191
EXPECTED_COLUMNS = ["date", "currency", "fx_rate"]


# 2. Load Raw Data

Read the complete source as text and validate required fields before conversion.


In [ ]:
raw = pl.read_csv(RAW_DIR / SOURCE_FILE, infer_schema=False, null_values=[""])
if raw.columns != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected FX source columns: {raw.columns}")
missing_required = {column: raw[column].null_count() for column in EXPECTED_COLUMNS if raw[column].null_count()}
if missing_required:
    raise ValueError(f"Required FX values are NULL: {missing_required}")
display(raw.head(5))


# 3. Cleaning & Typing

Store the date as Date and the rate as Decimal(18,6), retaining the source quote direction and currency labels.


In [ ]:
staged = raw.with_columns([
    pl.col("date").str.to_date(format="%Y-%m-%d", strict=False),
    pl.col("fx_rate").cast(pl.Decimal(18, 6), strict=False),
])
if staged["date"].null_count() or staged["fx_rate"].null_count():
    raise ValueError("Invalid FX date or rate encountered during parsing")
if staged.schema["date"] != pl.Date or staged.schema["fx_rate"] != pl.Decimal(18, 6):
    raise TypeError("Unexpected FX data types")
if staged.filter(pl.col("fx_rate") <= 0).height:
    raise ValueError("FX rates must be positive")
if set(staged["currency"].unique().to_list()) != {"EUR", "HUF"}:
    raise ValueError("Unexpected FX currency values")


# 4. Data Quality & Reconciliation

Reconcile the counts recorded for this supplied case-study export, validate unique date-currency keys, and preserve the original gaps in non-business days.


In [ ]:
if raw.height != CASE_STUDY_EXPECTED_ROWS:
    raise ValueError(f"Expected {CASE_STUDY_EXPECTED_ROWS} rows in the supplied case-study export, found {raw.height}")
if staged.height != raw.height:
    raise ValueError(f"FX rows were lost during staging: source={raw.height}, staged={staged.height}")
if staged.select(["date", "currency"]).n_unique() != staged.height:
    raise ValueError("FX date-currency keys are not unique")
counts = staged.group_by("currency").len()
if counts.filter(pl.col("len") != CASE_STUDY_EXPECTED_ROWS_PER_CURRENCY).height or counts.height != 2:
    raise ValueError(f"Unexpected FX records per currency: {counts}")
display(counts.sort("currency"))
display(staged.head(5))


# 5. Parquet Export

Export only observed FX records; no missing dates are synthesized.


In [ ]:
staged.write_parquet(OUTPUT_PATH)
exported = pl.read_parquet(OUTPUT_PATH)
if exported.schema != staged.schema:
    raise TypeError(f"FX Parquet schema differs from staged schema: {OUTPUT_PATH}")
if not exported.equals(staged):
    raise ValueError(f"FX Parquet contents differ from staged data: {OUTPUT_PATH}")
display(pl.DataFrame({"output_file": [OUTPUT_PATH.name], "rows": [exported.height], "schema": [str(exported.schema)]}))
